In [ ]:
import cv2
import numpy as np
from utils import extract_circle_mask

In [10]:
# Shi-Tomasi feature-detection settings
feature_params = {
    "maxCorners": 250,
    "qualityLevel": 0.2,
    "minDistance": 7,
    "blockSize": 7,
}

# Lucas-Kanade optical-flow settings
lk_params = {
    "winSize": (21, 21),
    "maxLevel": 3,
    "criteria": (
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        30,
        0.01,
    ),
}


In [11]:
input_video = "videos/p07_left_preop_scope02.mp4"

capture = cv2.VideoCapture(input_video)
if not capture.isOpened():
    raise RuntimeError(f"Could not open video: {input_video}")

fps = capture.get(cv2.CAP_PROP_FPS)
if not fps or np.isnan(fps):
    fps = 30.0

width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
duration_ms = (total_frames / fps) * 1000

print(f"Input video opened: {width} x {height} at {fps:.2f} FPS")
print(f"Total frames: {total_frames}")
print(f"Video duration: {duration_ms:.2f} ms")


Input video opened: 1920 x 1080 at 30.00 FPS
Total frames: 817
Video duration: 27233.33 ms


In [12]:
output_video = "videos/p07_left_preop_scope02_lk_with_mask.mp4"

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)


In [13]:
ok, previous_frame = capture.read()
if not ok:
    capture.release()
    writer.release()
    raise RuntimeError("Could not read the first video frame.")

circle_mask = extract_circle_mask(previous_frame)
previous_gray = cv2.cvtColor(previous_frame, cv2.COLOR_BGR2GRAY)
points_previous = cv2.goodFeaturesToTrack(
    previous_gray,
    mask=circle_mask,
    **feature_params,
)

trail_layer = np.zeros_like(previous_frame)
random_generator = np.random.default_rng(7)
colors = random_generator.integers(
    0,
    256,
    size=(feature_params["maxCorners"], 3),
)

frame_number = 1
tracked_point_total = 0
print(f"Initial features detected: {0 if points_previous is None else len(points_previous)}")


Obtaining circle mask...
Initial features detected: 15


In [14]:
while True:
    ok, current_frame = capture.read()
    if not ok:
        break

    current_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)

    if points_previous is not None and len(points_previous) > 0:
        points_current, status, _ = cv2.calcOpticalFlowPyrLK(
            previous_gray,
            current_gray,
            points_previous,
            None,
            **lk_params,
        )

        if points_current is not None and status is not None:
            valid = status.ravel() == 1
            good_current = points_current[valid]
            good_previous = points_previous[valid]
            tracked_point_total += len(good_current)

            for point_index, (current, previous) in enumerate(
                zip(good_current, good_previous)
            ):
                x_current, y_current = current.ravel().astype(int)
                x_previous, y_previous = previous.ravel().astype(int)
                color = tuple(
                    int(value) for value in colors[point_index % len(colors)]
                )
                cv2.line(
                    trail_layer,
                    (x_previous, y_previous),
                    (x_current, y_current),
                    color,
                    2,
                )
                cv2.circle(
                    current_frame,
                    (x_current, y_current),
                    4,
                    color,
                    -1,
                )

            points_previous = (
                good_current.reshape(-1, 1, 2)
                if len(good_current) > 0
                else None
            )
        else:
            points_previous = None

    if points_previous is None or len(points_previous) < 20:
        points_previous = cv2.goodFeaturesToTrack(
            current_gray,
            mask=circle_mask,
            **feature_params,
        )

    writer.write(cv2.add(current_frame, trail_layer))
    previous_gray = current_gray
    frame_number += 1


In [15]:
capture.release()
writer.release()
cv2.destroyAllWindows()

print(f"Processed {frame_number} frames")
print(f"Frame size: {width} x {height}")
print(f"Accumulated tracked points: {tracked_point_total}")
print(f"Output video saved as: {output_video}")


Processed 817 frames
Frame size: 1920 x 1080
Accumulated tracked points: 29344
Output video saved as: videos/p07_left_preop_scope02_lk_with_mask.mp4
